# Titanic 生存预测分析

## 1. 问题定义

**任务描述：** 预测泰坦尼克乘客是否生存（二分类问题）

**为什么选 F1-score 而不是 Accuracy？**
数据集中约62%的乘客死亡，类别不平衡。若模型全部预测"死亡"，Accuracy 仍有62%，但毫无意义。
F1-score 综合考虑精确率和召回率，对不平衡数据更有区分度。

**评估框架：**
- F1-score（主要指标）
- AUC-ROC（综合评估分类能力）
- 混淆矩阵（直观展示预测分布）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 从 seaborn 内置数据集加载（无需下载）
df = sns.load_dataset('titanic')
print(f"数据集形状: {df.shape}")
df.head()

## 2. 探索性数据分析（EDA）

### 2.1 缺失值分析

首先了解数据质量，缺失值的处理策略将在特征工程阶段决定。

In [ ]:
# 缺失值分析
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'缺失数量': missing, '缺失比例(%)': missing_pct})
missing_df[missing_df['缺失数量'] > 0].sort_values('缺失比例(%)', ascending=False)

### 2.2 生存率分布分析

观察关键特征与生存率的关系，为后续特征选择提供依据。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 总体生存率
df['survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'])
axes[0].set_title('总体生存分布
(0=死亡, 1=生存)')
axes[0].set_xlabel('')

# 性别与生存率
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[1], color=['#3498db','#e91e63'])
axes[1].set_title('性别与生存率')
axes[1].set_ylabel('生存率')

# 舱位等级与生存率
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[2], color=['#f39c12','#95a5a6','#7f8c8d'])
axes[2].set_title('舱位等级与生存率')
axes[2].set_ylabel('生存率')

plt.tight_layout()
plt.savefig('eda_survival.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 EDA 关键发现

- **性别影响显著：** 女性生存率约74%，男性约19%（"女士优先"原则体现）
- **舱位等级影响显著：** 一等舱生存率约63%，三等舱约24%（经济地位影响逃生机会）
- **缺失值情况：** `age` 缺失约20%，`deck` 缺失约77%，`embarked` 缺失2条

这些发现直接指导了后续的特征工程决策。